# 02 — Calibration Analysis

Validates GLFT and Cartea-Jaimungal calibrators against real tick data from DuckDB,
then runs the full §8 framework: Brier decomposition + recalibration comparison.

Sections:
1. Setup & data loading
2. Exploratory data analysis
3. GLFT calibration
4. CJ calibration
5. Brier decomposition (§8.1)
6. Recalibration comparison
7. Signal ensemble (§8.3)
8. Parameter update to YAML

In [1]:
import sys, os
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="darkgrid", palette="tab10")
plt.rcParams["figure.figsize"] = (12, 4)
print("Project root:", ROOT)

Project root: /home/acho/PredMkts/prediction-market-system


In [2]:
from storage.reader import MarketDataReader
from features.microstructure import ewma_vol_series, obi_series
from features.calibration import (
    brier_decompose,
    VennAbersCalibrator,
    IsotonicCalibrator,
    BetaCalibrator,
)
from strategies.market_making.params import GLFTCalibrator, CJCalibrator
from features.signals.ensemble import SignalEnsemble

reader = MarketDataReader()
print("Reader ready")

ValidationError: 1 validation error for Settings
kalshi_env
  String should match pattern '^(demo|prod)$' [type=string_pattern_mismatch, input_value='demo          # demo | prod', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/string_pattern_mismatch

## 1. Data loading

In [ ]:
markets = reader.list_markets()
print(f"{len(markets)} markets in DuckDB")
markets[["market_id", "venue", "title", "status"]].head(20)

In [ ]:
tick_counts = {}
for mid in markets["market_id"]:
    df = reader.latest_ticks(mid, n=10_000)
    tick_counts[mid] = len(df)
tick_counts = pd.Series(tick_counts).sort_values(ascending=False)
MARKET_ID = tick_counts.index[0]
print(f"Using: {MARKET_ID}  ({tick_counts[MARKET_ID]} ticks)")

In [ ]:
ticks_df = reader.latest_ticks(MARKET_ID, n=10_000).sort_values("timestamp").reset_index(drop=True)
features_df = reader.latest_features(MARKET_ID).sort_values("timestamp").reset_index(drop=True)
print(f"Ticks: {len(ticks_df):,}  Features: {len(features_df):,}")
ticks_df.head()

## 2. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(ticks_df["timestamp"], ticks_df["mid"], lw=0.8)
axes[0].set_title("Mid price")
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
vol_series = ewma_vol_series(ticks_df)
axes[1].plot(ticks_df["timestamp"], vol_series, lw=0.8, color="tab:orange")
axes[1].set_title("EWMA belief vol sigma_b")
obi = obi_series(ticks_df, window=50)
axes[2].fill_between(ticks_df["timestamp"], obi, alpha=0.5, color="tab:green")
axes[2].axhline(0, color="k", lw=0.5)
axes[2].set_title("OBI (50-tick)")
for ax in axes:
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

## 3. GLFT calibration — MLE for kappa and A

Section 3.2 MATH.md v2.1

In [ ]:
if "spread" not in ticks_df.columns:
    if "yes_bid" in ticks_df.columns and "yes_ask" in ticks_df.columns:
        ticks_df["spread"] = ticks_df["yes_ask"] - ticks_df["yes_bid"]
    else:
        ticks_df["spread"] = 0.02
glft_cal = GLFTCalibrator()
glft_result = glft_cal.fit(ticks_df)
print(glft_result.summary())

In [ ]:
spreads_grid = np.linspace(
    ticks_df["spread"].quantile(0.05), ticks_df["spread"].quantile(0.95), 100
)
arrival_rate = glft_result.A * np.exp(-glft_result.kappa_p * spreads_grid)
plt.figure(figsize=(8, 4))
plt.plot(spreads_grid, arrival_rate)
plt.xlabel("Spread delta")
plt.ylabel("Arrival rate lambda")
plt.title(f"GLFT fill-rate  kappa={glft_result.kappa_p:.3f}  A={glft_result.A:.4f}")
plt.tight_layout()
plt.show()

## 4. CJ calibration — Ridge + AR(1) for phi, eta, rho, w_i

Section 4.6 MATH.md v2.1

In [ ]:
if features_df.empty or "obi" not in features_df.columns:
    features_df = ticks_df[["timestamp"]].copy()
    features_df["obi"] = obi_series(ticks_df, window=20).values
cj_cal = CJCalibrator(ridge_alpha=1.0)
cj_result = cj_cal.fit(ticks_df, features_df)
print(cj_result.summary())

In [ ]:
ensemble = SignalEnsemble.from_cj_result(cj_result)
mu_hat = features_df["obi"].apply(lambda x: ensemble.compute_mu_hat(obi=x))
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(ticks_df["timestamp"], ticks_df["mid"], lw=0.8)
axes[0].set_title("Price and CJ signal mu_t")
axes[1].plot(features_df["timestamp"], mu_hat, lw=0.7, color="tab:orange", alpha=0.8)
axes[1].axhline(0, color="k", lw=0.5, ls="--")
axes[1].set_ylabel("mu_hat")
axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()
print(f"Signal R2={cj_result.signal_r2:.4f}  AR1 alpha={cj_result.ar1_alpha:.4f}")

## 5. Brier decomposition (§8.1 MATH.md v2.1)

Br = REL - RES + UNC + WBV - 2*WBC

In [ ]:
df_eval = ticks_df[["timestamp", "mid"]].dropna().reset_index(drop=True)
df_eval["forecast"] = df_eval["mid"].shift(1)
df_eval["outcome"] = (df_eval["mid"] > df_eval["mid"].shift(1)).astype(float)
df_eval = df_eval.dropna().reset_index(drop=True)
print(f"Eval pairs: {len(df_eval):,}  Base rate: {df_eval['outcome'].mean():.4f}")
brier = brier_decompose(
    forecasts=df_eval["forecast"].values,
    outcomes=df_eval["outcome"].values,
    n_bins=20,
    bootstrap_n=200,
    random_state=42,
)
print()
print(brier.summary())

In [ ]:
from sklearn.calibration import calibration_curve

frac_pos, mean_pred = calibration_curve(
    df_eval["outcome"], df_eval["forecast"], n_bins=20, strategy="quantile"
)
plt.figure(figsize=(6, 6))
plt.plot([0, 1], [0, 1], "k--", lw=0.8, label="Perfect")
plt.plot(mean_pred, frac_pos, "o-", ms=5, label="Model")
plt.xlabel("Mean predicted")
plt.ylabel("Fraction positive")
plt.title(f"Reliability diagram  REL={brier.reliability:.4f}")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Recalibration comparison

Section 8.2: Venn-Abers vs Isotonic vs Beta, out-of-time split.

In [ ]:
n = len(df_eval)
split = int(n * 0.7)
train_f = df_eval["forecast"].values[:split]
train_o = df_eval["outcome"].values[:split]
test_f = df_eval["forecast"].values[split:]
test_o = df_eval["outcome"].values[split:]
calibrators = {
    "Raw": test_f,
    "Isotonic": IsotonicCalibrator().fit(train_f, train_o).predict(test_f),
    "Beta": BetaCalibrator().fit(train_f, train_o).predict(test_f),
    "Venn-Abers": VennAbersCalibrator().fit(train_f, train_o).predict(test_f),
}
rows = []
for name, preds in calibrators.items():
    preds = np.clip(preds, 1e-7, 1 - 1e-7)
    rows.append(
        {
            "Method": name,
            "Brier": float(np.mean((preds - test_o) ** 2)),
            "LogLoss": float(-np.mean(test_o * np.log(preds) + (1 - test_o) * np.log(1 - preds))),
        }
    )
pd.DataFrame(rows).set_index("Method").round(6)

## 7. Signal ensemble — Cholesky orthogonalization (§8.3)

News and on-chain stubs return 0.0. Section shows the Cholesky path.

In [ ]:
n_hist = min(len(features_df), 1000)
obi_vals = features_df["obi"].values[:n_hist]
rng = np.random.default_rng(0)
news_vals = 0.3 * obi_vals + 0.7 * rng.standard_normal(n_hist) * 0.1
onchain_vals = rng.standard_normal(n_hist) * 0.05
signal_matrix = np.column_stack([obi_vals, news_vals, onchain_vals])
print("Signal correlations:")
pd.DataFrame(signal_matrix, columns=["OBI", "News", "OnChain"]).corr().round(3)

In [ ]:
ens_ortho = SignalEnsemble.from_cj_result(cj_result)
ens_ortho.fit_orthogonalization(signal_matrix)
mu_raw = obi_vals * cj_result.w_obi
mu_ortho = np.array(
    [ens_ortho.compute_mu_hat(o, n, c) for o, n, c in zip(obi_vals, news_vals, onchain_vals)]
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(mu_raw, lw=0.8, label="raw")
axes[0].plot(mu_ortho, lw=0.8, alpha=0.7, label="ortho")
axes[0].legend()
axes[0].set_title("mu_hat: raw vs Cholesky-orthogonalized")
axes[1].scatter(mu_raw, mu_ortho, alpha=0.3, s=5)
axes[1].plot([mu_raw.min(), mu_raw.max()], [mu_raw.min(), mu_raw.max()], "r--", lw=0.8)
axes[1].set_title(f"Correlation: {np.corrcoef(mu_raw, mu_ortho)[0, 1]:.3f}")
plt.tight_layout()
plt.show()

## 8. Parameter update to YAML configs

In [ ]:
venue = MARKET_ID.split(":")[0] if ":" in MARKET_ID else "kalshi"
new_params = {**glft_result.to_dict(), **cj_result.to_dict()}
print(f"Parameters for venue={venue}:")
for k, v in new_params.items():
    print(f"  {k:12s} = {v:.6f}")
# Uncomment to persist:
# update_model_params(venue, new_params)
print("(update commented — review before persisting)")

## Summary

| Component | Status | Notes |
|-----------|--------|-------|
| GLFT calibration | done | kappa_p, kappa_x, A from MLE |
| CJ calibration | done | phi, eta, rho, w_obi from Ridge+AR(1) |
| Brier decomposition | done | 5-component with WBV, WBC, bootstrap CI |
| Venn-Abers | done | Out-of-time split, valid coverage |
| Ensemble | ready | Cholesky path wired; waiting for real news/onchain |